In [ ]:
import os 
from patient import Patient
import pandas as pd
import numpy as np
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler


In [5]:
MAIN = r"C:\Users\timje\Documents\dataset\data_final\I0002"

rng = np.random.default_rng(seed=42)

In [ ]:
def generate_feature_dataframes(only_these=None):
    """Instantiates all patients and thereby the feature dataframe for each of the patients. The filepaths for the dataframes are 
    stored for later usage.

    If a parameter list only_these is given, the dataframe will only be generated for them.   

    """

    if only_these == None:
        subjects = [name for name in os.listdir(MAIN) if os.path.isdir(os.path.join(MAIN, name))]
    else:
        subjects = only_these

    paths_dataframes = []
    failed_subjects = []

    for i, subject in enumerate(subjects): 

        #create patient object, which builds the feature dataframe 
        try: 
            patient = Patient(id=subject, file_path=MAIN)
            #subject_objects.append(patient)

            # take the file path of the feature dataframe for only the first session, to prevent bias
            filepath = patient.file_paths_feature_dataframes[0]
            paths_dataframes.append(filepath)

            print(f"Generated {i}th subject. Id: {subject}")
        except:
            print(f"Failed to generate subject {subject}.")
            failed_subjects.append(subject)


    # store paths permanently
    lines = [path + '\n' for path in paths_dataframes]
    with open('paths.txt', 'w') as f:
        f.writelines(lines)

    return paths_dataframes, failed_subjects

In [ ]:
# retrieve all parquet files again
subjects = [name for name in os.listdir(MAIN) if os.path.isdir(os.path.join(MAIN, name))]
paths_dataframes = []
no_dataframe_exists = []

for subject in subjects:
    path = os.path.join(MAIN, subject, "ses-1", "eeg", f"{subject}_ses-1_feature_dataframe.parquet")
    if os.path.isfile(path):
        paths_dataframes.append(path)
    else:
        print(f"For subject {subject} no parquet file exists.")
        no_dataframe_exists.append(subject)

        

['C:\\Users\\timje\\Documents\\dataset\\data_final\\I0002\\sub-I0002150000375\\ses-1\\eeg\\sub-I0002150000375_ses-1_feature_dataframe.parquet', 'C:\\Users\\timje\\Documents\\dataset\\data_final\\I0002\\sub-I0002150000580\\ses-1\\eeg\\sub-I0002150000580_ses-1_feature_dataframe.parquet']
800
0


In [5]:
# check wheter all subjects do have a fif file
subjects = [name for name in os.listdir(MAIN) if os.path.isdir(os.path.join(MAIN, name))]
fif_file_exits = []
fif_file_does_not_exit = []

for subject in subjects:
    path = os.path.join(MAIN, subject, "ses-1", "eeg", f"{subject}_ses-1_task-PSG_eeg.fif")
    if os.path.isfile(path):
        fif_file_exits.append(subject)
    else:
        print(f"For subject {subject} no fif file exists.")
        fif_file_does_not_exit.append(subject)

print(len(fif_file_exits))
print(len(fif_file_does_not_exit))

800
0


In [ ]:
paths_dataframes, failed = generate_feature_dataframes()


In [ ]:
def generate_dataset(filepaths):

    """Generates the complete dataset for training and testing. Contains Apnea, Hypopnea and Normal episodes in the proportions described in the hyperparameters.
    Takes as parameter a list of the file paths of the feature dataframes. 
    
    Returns the dataset and simultaneosuly stores parquet and csv files.
    """ 

    NORMAL_PERCENTAGE = 0.33
    HYPOPNEA_PERCENTAGE = 0.33
    APNEA_PERCENTAGE = 0.33
    MAX_EPSD_PER_SUBJ = 40

    max_extra = 20

    collected_rows = []
    missing_apnea = 0
    extras_apnea = []
    missing_hypopnea = 0
    extras_hypopnea = []
    missing_normal = 0
    extras_normal = []

    weights_apnea = []
    weights_hypopnea = []
    weights_normal = []

    for path in filepaths:

        df = pd.read_parquet(path)
        ahi = df["AHI"].values[0]

        if ahi < 5:
            weight = 20
        elif ahi >= 5 and ahi < 15:
            weight = 10
        elif ahi >= 15 and ahi < 30:
            weight = 3
        else:
            weight = 1 

        # --- apnea episodes ---
        max_apnea_episodes = int(APNEA_PERCENTAGE * MAX_EPSD_PER_SUBJ)
        apnea_indices = df.index[(df['Apnea'] == "Apnea") & (~df["Smini"].isna()) & (~df["rr_mean"].isna())].tolist()

        if len(apnea_indices) <= max_apnea_episodes:
            rows_dicts = df.iloc[apnea_indices].to_dict('records')
            missing_apnea += max_apnea_episodes-len(apnea_indices)
        else:
            selected_episodes = rng.choice(apnea_indices, size=max_apnea_episodes, replace=False)
            rows_dicts = df.iloc[selected_episodes].to_dict('records')

            # store the extra episodes in case they are needed (maximum of 20)
            extra_episodes  = np.setdiff1d(apnea_indices, selected_episodes)
            extra_dicts = df.iloc[extra_episodes].to_dict('records')
            extra_dicts = extra_dicts if len(extra_dicts) < max_extra else extra_dicts[:max_extra]
            extras_apnea.extend(extra_dicts)

            weights_apnea.extend([weight]*len(extra_dicts))
            

        collected_rows.extend(rows_dicts)
        

        # --- hypopnea episodes ---
        max_hypopnea_episodes = int(HYPOPNEA_PERCENTAGE * MAX_EPSD_PER_SUBJ) 
        hypopnea_indices = df.index[(df['Apnea'] == "Hypopnea") & (~df["Smini"].isna()) & (~df["rr_mean"].isna())].tolist()

        if len(hypopnea_indices) <= max_hypopnea_episodes:
            rows_dicts = df.iloc[hypopnea_indices].to_dict('records')
            missing_hypopnea += max_hypopnea_episodes-len(hypopnea_indices)
        else:
            selected_episodes = rng.choice(hypopnea_indices, size=max_hypopnea_episodes, replace=False)
            rows_dicts = df.iloc[selected_episodes].to_dict('records')

            # store the extra episodes in case the are needed
            extra_episodes  = np.setdiff1d(hypopnea_indices, selected_episodes)
            extra_dicts = df.iloc[extra_episodes].to_dict('records')
            extra_dicts = extra_dicts if len(extra_dicts) < max_extra else extra_dicts[:max_extra]
            extras_hypopnea.extend(extra_dicts)

            weights_hypopnea.extend([weight]*len(extra_dicts))

        collected_rows.extend(rows_dicts)
        


        # --- normal episodes ---
        max_normal_episodes = int(NORMAL_PERCENTAGE * MAX_EPSD_PER_SUBJ)
        normal_indices = df.index[(df['Apnea'] == "Normal") & (~df["Smini"].isna()) & (~df["rr_mean"].isna())].tolist()

        if len(normal_indices) <= max_normal_episodes:
            rows_dicts = df.iloc[normal_indices].to_dict('records')
            missing_normal += max_normal_episodes- len(normal_indices)
        else:
            selected_episodes = rng.choice(normal_indices, size=max_normal_episodes, replace=False)
            rows_dicts = df.iloc[selected_episodes].to_dict('records')

            # store the extra episodes in case the are needed
            extra_episodes  = np.setdiff1d(normal_indices, selected_episodes)
            extra_dicts = df.iloc[extra_episodes].to_dict('records')
            extra_dicts = extra_dicts if len(extra_dicts) < max_extra else extra_dicts[:max_extra]
            extras_normal.extend(extra_dicts)

            weights_normal.extend([weight]*len(extra_dicts))

        collected_rows.extend(rows_dicts)

    weights_apnea = np.array(weights_apnea)
    weights_apnea = list(weights_apnea / np.sum(weights_apnea))

    weights_hypopnea = np.array(weights_hypopnea)
    weights_hypopnea = list(weights_hypopnea / np.sum(weights_hypopnea))

    weights_normal = np.array(weights_normal)
    weights_normal = list(weights_normal / np.sum(weights_normal))

    if missing_apnea != 0:
        idx = np.arange(len(extras_apnea))
        selected_episodes = rng.choice(idx, size=min(len(extras_apnea), missing_apnea), p=weights_apnea, replace=False)
        print(f"Finally {len(selected_episodes)-missing_apnea} apnea episodes are missing.")
        rows_dicts = [extras_apnea[i] for i in selected_episodes]
        collected_rows.extend(rows_dicts)

    if missing_hypopnea != 0:
        idx = np.arange(len(extras_hypopnea))
        selected_episodes = rng.choice(idx, size=min(len(extras_hypopnea), missing_hypopnea), p=weights_hypopnea, replace=False)
        print(f"Finally {len(selected_episodes)-missing_hypopnea} hypopnea episodes are missing.")
        rows_dicts = [extras_hypopnea[i] for i in selected_episodes]
        collected_rows.extend(rows_dicts)

    if missing_normal != 0:
        idx = np.arange(len(extras_normal))
        selected_episodes = rng.choice(idx, size=min(len(extras_normal), missing_normal), p=weights_normal, replace=False)
        print(f"Finally {len(selected_episodes)-missing_normal} hypopnea episodes are missing.")
        rows_dicts = [extras_normal[i] for i in selected_episodes]
        collected_rows.extend(rows_dicts)



    
    df = pd.DataFrame(collected_rows)
    dataset = df.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle data by sampling all rows; fix 
    dataset.to_parquet(f"final_datasets\\TEST_dataset_A{int(APNEA_PERCENTAGE*100)}_H{int(HYPOPNEA_PERCENTAGE*100)}_N{int(NORMAL_PERCENTAGE*100)}.parquet")
    dataset.to_csv(f"final_datasets\\TEST_dataset_A{int(APNEA_PERCENTAGE*100)}_H{int(HYPOPNEA_PERCENTAGE*100)}_N{int(NORMAL_PERCENTAGE*100)}.csv")

    return dataset


In [9]:
dataset = generate_dataset(paths_dataframes)

Finally 0 apnea episodes are missing.
Finally 0 hypopnea episodes are missing.
Finally 0 hypopnea episodes are missing.


In [17]:
# calcualte metrices of the dataset 
dataset = pd.read_parquet(r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A33_H33_N33.parquet")


subjects = np.array(dataset["Subject"].unique())
print(f"Number of unique subjects: {len(subjects)}")

ahi = np.array([dataset[dataset["Subject"] == subject].iloc[0]["AHI"] for subject in subjects])
normal = sum((ahi < 5).astype(int))
mild = sum(((ahi >= 5) & (ahi < 15)).astype(int))
moderate = sum(((ahi >= 15) & (ahi < 30)).astype(int))
severe = sum((ahi >= 30).astype(int))

print(f"AHI normal: {normal}, mild {mild}, moderate {moderate} and severe {severe}")


df_unique = dataset.drop_duplicates(subset=["Subject"])

ages_men = df_unique[df_unique["Sex"] == "M"]["Age"].values
ages_women = df_unique[df_unique["Sex"] == "F"]["Age"].values
ages = df_unique["Age"].values
average_age = np.mean(ages)
average_age_men = np.mean(ages_men)
average_age_women = np.mean(ages_women)

print(f"Average age is {average_age} with {average_age_men} for men and {average_age_women} for women.")

sex = np.array([dataset[dataset["Subject"] == subject].iloc[0]["Sex"] for subject in subjects])
men = np.sum((sex == "M").astype(int))
women = np.sum((sex == "F").astype(int))

print(f"There are {men} men ({men / len(sex)*100}%) and {women} ({women/len(sex)*100}%) women. ")


Number of unique subjects: 800
AHI normal: 200, mild 200, moderate 200 and severe 200
Average age is 51.72125 with 51.34413965087282 for men and 52.100250626566414 for women.
There are 401 men (50.125%) and 399 (49.875%) women. 


In [ ]:
# get statistics about dataset

dataset = pd.read_parquet(r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet")

# general sex distribution
sex_counts = dataset["Sex"].value_counts()
totalsex = len(dataset["Sex"])
print("--- General Sex Distribution ---")
print(f"Male:   {sex_counts['M']} ({sex_counts['M']/totalsex*100:.2f}%)")
print(f"Female: {sex_counts['F']} ({sex_counts['F']/totalsex*100:.2f}%)")

# subject statistics
counts = dataset["Subject"].value_counts().values
print("\n--- Episodes per Subject ---")
print(f"Mean: {np.mean(counts):.2f}, Median: {np.median(counts)}, SD: {np.std(counts):.2f}, Range: [{np.min(counts)}, {np.max(counts)}]")


categories = ["Apnea", "Hypopnea", "Normal"]
severity_levels = {
    "Normal": (lambda x: x < 5),
    "Mild": (lambda x: (x >= 5) & (x < 15)),
    "Moderate": (lambda x: (x >= 15) & (x < 30)),
    "Severe": (lambda x: x >= 30)
}

table_data = []

for cat in categories:
    row = {"Category": cat}
    cat_total = len(dataset[dataset["Apnea"] == cat])
    
    for sev_name, sev_cond in severity_levels.items():

        subset = dataset[(dataset["Apnea"] == cat) & (sev_cond(dataset["AHI"]))]
        
        count = len(subset)
        perc = (count / cat_total * 100) if cat_total > 0 else 0
        women_count = len(subset[subset["Sex"] == "F"])
        women_perc = women_count / len(subset) * 100 if len(subset) > 0 else 0
        
        row[sev_name] = f"{count} ({perc:.1f}%)"
        row[f"{sev_name}_F"] = women_count

        row[f"{sev_name}_F_%"] = f"{women_perc:.1f}"
        
    table_data.append(row)

df_results = pd.DataFrame(table_data)

main_table = df_results[["Category", "Normal", "Mild", "Moderate", "Severe"]]
print("\nTable 4.1: Distribution of Estimated Sleep Apnea Severity Levels")
print(main_table.to_string(index=False))

women_table = df_results[["Category", "Normal_F", "Normal_F_%","Mild_F", "Mild_F_%", "Moderate_F", "Moderate_F_%","Severe_F", "Severe_F_%"]]
women_table.columns = ["Category", "Normal (F)", "Normal_%", "Mild (F)", "Mild %", "Moderate (F)", "Moderate %", "Severe (F)", "Severe %"]
print("\nTable 4.2: Distribution of Episodes among Women")
print(women_table.to_string(index=False))

--- General Sex Distribution ---
Male:   16317 (50.99%)
Female: 15683 (49.01%)

--- Episodes per Subject ---
Mean: 40.00, Median: 39.0, SD: 9.46, Range: [2, 54]

Table 4.1: Distribution of Estimated Sleep Apnea Severity Levels
Category       Normal         Mild     Moderate       Severe
   Apnea   520 (6.5%) 1830 (22.9%) 2958 (37.0%) 2692 (33.7%)
Hypopnea 2016 (25.2%) 2223 (27.8%) 2038 (25.5%) 1723 (21.5%)
  Normal 4074 (25.5%) 4037 (25.2%) 4013 (25.1%) 3876 (24.2%)

Table 4.2: Distribution of Episodes among Women
Category  Normal (F) Normal_%  Mild (F) Mild %  Moderate (F) Moderate %  Severe (F) Severe %
   Apnea         251     48.3       827   45.2          1370       46.3        1311     48.7
Hypopnea         979     48.6      1110   49.9          1021       50.1         845     49.0
  Normal        2038     50.0      2018   50.0          2007       50.0        1906     49.2

Detailed metrics exported to 'dataset_metrics.csv'


In [ ]:
# get csv for every parquet file for visualization purposes

for dirpath, dirnames, filenames in os.walk(r"C:\Users\timje\Documents\dataset\balanced_data\I0002"):
        for filename in filenames:
            if filename.endswith('.parquet'):
                # Construct the full path
                full_path = os.path.join(dirpath, filename)

                df = pd.read_parquet(full_path)

                base_path, ext = os.path.splitext(full_path)
                full_path_csv = base_path + ".csv"

                df.to_csv(full_path_csv, index=False) 